In [1]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import evaluate
import EIDA


model_name = "roberta-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)


max_length = 72
batch_size = 16
dataset = load_dataset("glue", "sst2")
def preprocess_function(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )
tokenized_dataset = dataset.map(preprocess_function, batched=True)

train_dataset = tokenized_dataset['train']
dev_dataset = tokenized_dataset['validation']


metric = evaluate.load("glue", "sst2")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=1)
    return metric.compute(predictions=predictions, references=labels)

c:\Users\3un8i\anaconda3\envs\IDA\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
for name, param in model.roberta.embeddings.named_parameters():
    param.requires_grad = False
for layer in model.roberta.encoder.layer:
    layer.attention.output.LayerNorm.weight.requires_grad = False
    layer.attention.output.LayerNorm.bias.requires_grad = False
    layer.output.LayerNorm.weight.requires_grad = False
    layer.output.LayerNorm.bias.requires_grad = False


for _ in range(5):
    train_dataset = train_dataset.shuffle()
    input_ids = torch.tensor(train_dataset['input_ids'])
    attention_mask = torch.tensor(train_dataset['attention_mask'])
    label = torch.tensor(train_dataset['label'])
    
    sample_inputs, sample_delta_outputs = EIDA.forward(model, input_ids, attention_mask, label, begin=0, end=256, batch_size=batch_size, max_length=max_length, N=2)

    plane_inputs=[]
    for i in range(4*12+1):
        plane_inputs.append(EIDA.PCA(sample_inputs[i], plane_dim=32))
    del sample_inputs

    plane_delta_outputs=[]
    for i in range(6*12+1):
        plane_delta_outputs.append(EIDA.PCA(sample_delta_outputs[i], plane_dim=32))
    del sample_delta_outputs

    for l, layer in enumerate(model.roberta.encoder.layer):
        layer.attention.self.query = EIDA.Linear_with_adapter(layer.attention.self.query, plane_inputs[4*l+0], plane_delta_outputs[6*l+0])
        layer.attention.self.key = EIDA.Linear_with_adapter(layer.attention.self.key, plane_inputs[4*l+0], plane_delta_outputs[6*l+1])
        layer.attention.self.value = EIDA.Linear_with_adapter(layer.attention.self.value, plane_inputs[4*l+0], plane_delta_outputs[6*l+2])
        layer.attention.output.dense = EIDA.Linear_with_adapter(layer.attention.output.dense, plane_inputs[4*l+1], plane_delta_outputs[6*l+3])
        layer.intermediate.dense = EIDA.Linear_with_adapter(layer.intermediate.dense, plane_inputs[4*l+2], plane_delta_outputs[6*l+4])
        layer.output.dense = EIDA.Linear_with_adapter(layer.output.dense, plane_inputs[4*l+3], plane_delta_outputs[6*l+5])
    model.classifier.dense = EIDA.Linear_with_adapter(model.classifier.dense, plane_inputs[4*12+0], plane_delta_outputs[6*12+0])


    training_args = TrainingArguments(
        output_dir=os.path.join("results", "sst2"),
        save_strategy="no",
        learning_rate=1e-4,
        warmup_ratio=0.1,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=1,
        weight_decay=0.1,
        logging_dir="./logs",
        logging_steps=300,
        eval_strategy="steps",
        eval_steps=300,
        bf16=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    for l, layer in enumerate(model.roberta.encoder.layer):
        layer.attention.self.query = layer.attention.self.query.merge()
        layer.attention.self.key = layer.attention.self.key.merge()
        layer.attention.self.value = layer.attention.self.value.merge()
        layer.attention.output.dense = layer.attention.output.dense.merge()
        layer.intermediate.dense = layer.intermediate.dense.merge()
        layer.output.dense = layer.output.dense.merge()
    model.classifier.dense = model.classifier.dense.merge()

    model.save_pretrained(os.path.join("results", "sst2"))

C:\Users\3un8i\AppData\Local\Temp\ipykernel_4440\1510371549.py:54: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
  7%|▋         | 300/4210 [00:17<03:40, 17.71it/s]

{'loss': 0.6897, 'grad_norm': 0.6029708385467529, 'learning_rate': 7.125890736342043e-05, 'epoch': 0.07}


                                                  
  7%|▋         | 303/4210 [00:19<12:29,  5.22it/s]

{'eval_loss': 0.6950750946998596, 'eval_accuracy': 0.5091743119266054, 'eval_runtime': 1.2772, 'eval_samples_per_second': 682.751, 'eval_steps_per_second': 43.063, 'epoch': 0.07}


 14%|█▍        | 600/4210 [00:36<03:33, 16.89it/s]

...

{'loss': 0.2178, 'grad_norm': 7.3033952713012695, 'learning_rate': 2.639218791237794e-07, 'epoch': 1.0}



100%|█████████▉| 4202/4210 [04:12<00:01,  4.09it/s]

{'eval_loss': 0.239402636885643, 'eval_accuracy': 0.9254587155963303, 'eval_runtime': 1.2639, 'eval_samples_per_second': 689.95, 'eval_steps_per_second': 43.517, 'epoch': 1.0}


100%|██████████| 4210/4210 [04:12<00:00, 16.67it/s]


{'train_runtime': 252.6171, 'train_samples_per_second': 266.605, 'train_steps_per_second': 16.666, 'train_loss': 0.2231070733127005, 'epoch': 1.0}
